Con riferimento al data set dell’esercitazione su clustering, eseguire una classificazione binaria sulla feature death, una classificazione multiclasse sulla feature dzgroup ed una regressione sulla feature aps.

**Regressione su "aps"**

1. Procedere allo split train-test secondo il rapporto 95%-5% in forma stratificata secondo i valori della variabile target.

In [60]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

df = pd.read_csv(".\dataset_esercitazione.csv", sep=',')
print(df)
df = df.dropna(subset=['aps']).copy()
y = df['aps']  #variabile target continua, a diff. di 'death' e 'dzgroup' che sono discrete

#stratificazione di una variabile continua
#bins = np.linspace(y.min(), y.max(), 5)  #suddividiamo 'aps' in 5 fasce (bins) per permettere a sklearn di stratificare
#y_binned = np.digitize(y, bins)

# Utilizziamo i quantili per dividere i pazienti in 5 fasce di uguale numerosità
y_binned = pd.qcut(y, q=4, labels=False, duplicates='drop')

'''
colonne_da_droppare = [
    'aps',      # Il nostro target
    'death',    # Outcome futuro (target della task precedente)
    'surv2m',   # Sopravvivenza a 2 mesi
    'surv6m',   # Sopravvivenza a 6 mesi
    'prg2m',    # Prognosi a 2 mesi
    'prg6m',    # Prognosi a 6 mesi
    'dnr',      # Do Not Resuscitate status (spesso deciso post-ammissione)
    'dnrday'    # Giorno della decisione DNR
]
NOTA METODOLOGICA SULLA SELEZIONE DELLE FEATURE:
Ho rimosso le variabili di outcome futuro per evitare il fenomeno del Target/Data Leakage. 
Poiché il target 'aps' (APACHE score) misura la gravità della malattia all'ingresso, 
includere informazioni sulla sopravvivenza o sulla morte del paziente (avvenute successivamente) 
altererebbe artificialmente le prestazioni del modello, rendendolo inutilizzabile in un contesto reale.
'''
X = df.drop(columns=['aps', 'sps'])  #le droppo entrambe perché fortemente correlate


X_train, X_test, y_train, y_test = train_test_split(
    X, 
    y, 
    test_size=0.05,  #5% per il test set
    stratify=y_binned,  #usiamo i bin per stratificare proporzionalmente
    random_state=42  #seed per la riproducibilità
)

print(f"Dimensioni X_train: {X_train.shape}")
print(f"Dimensioni X_test: {X_test.shape}")

           age     sex            dzgroup             dzclass  num.co   edu  \
0     62.84998    male        Lung Cancer              Cancer       0  11.0   
1     60.33899  female          Cirrhosis  COPD/CHF/Cirrhosis       2  12.0   
2     52.74698  female          Cirrhosis  COPD/CHF/Cirrhosis       2  12.0   
3     42.38498  female        Lung Cancer              Cancer       2  11.0   
4     79.88495  female  ARF/MOSF w/Sepsis            ARF/MOSF       1   NaN   
...        ...     ...                ...                 ...     ...   ...   
9100  66.07300    male  ARF/MOSF w/Sepsis            ARF/MOSF       1   8.0   
9101  55.15399  female               Coma                Coma       1  11.0   
9102  70.38196    male  ARF/MOSF w/Sepsis            ARF/MOSF       1   NaN   
9103  47.01999    male       MOSF w/Malig            ARF/MOSF       1  13.0   
9104  81.53894  female  ARF/MOSF w/Sepsis            ARF/MOSF       1   8.0   

          income  scoma  charges      totcst  ...  

<>:5: SyntaxWarning: invalid escape sequence '\d'
<>:5: SyntaxWarning: invalid escape sequence '\d'
C:\Users\giova\AppData\Local\Temp\ipykernel_40276\1677236611.py:5: SyntaxWarning: invalid escape sequence '\d'
  df = pd.read_csv(".\dataset_esercitazione.csv", sep=',')


2. Eseguire l’imputazione dei dati mancanti con le stesse strategie dell’esercitazione precedente.

In [61]:
from sklearn.preprocessing import OrdinalEncoder
from sklearn.impute import SimpleImputer

cat_values = X_train.select_dtypes(include=['object', 'bool']).columns.tolist()
num_values = X_train.select_dtypes(include=['number']).columns.tolist()
print(cat_values)
#cat_values = [col for col in X_train.columns if col not in num_values]

#per gestire i dati mancanti
report_missing = X_train.isnull().sum() / len(X_train) * 100
print(f"Percentuale dati mancanti:\n{report_missing}")

#contiamo le colonne con i dati mancanti
nan_columns = report_missing[report_missing > 0].index.tolist()
print(f"Colonne con dati mancanti:{len(nan_columns)}")
missing_cat = [col for col in nan_columns if col in cat_values]
missing_num = [col for col in nan_columns if col in num_values]

#imputazione per variabili numeriche e categoriche
if len(missing_num) > 0:
    imputer_num = SimpleImputer(strategy='median')
    X_train[missing_num] = imputer_num.fit_transform(X_train[missing_num])
    X_test[missing_num] = imputer_num.transform(X_test[missing_num])
if len(missing_cat) > 0:
    imputer_cat = SimpleImputer(strategy='constant', fill_value='Unknown')
    X_train[missing_cat] = imputer_cat.fit_transform(X_train[missing_cat])
    X_test[missing_cat] = imputer_cat.transform(X_test[missing_cat])

#encoding per variabili categoriche
if len(cat_values) > 0:
    encoder_cat = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-5)
    X_train[cat_values] = encoder_cat.fit_transform(X_train[cat_values])
    X_test[cat_values] = encoder_cat.transform(X_test[cat_values])

['sex', 'dzgroup', 'dzclass', 'income', 'race', 'ca', 'dnr']
Percentuale dati mancanti:
age          0.000000
sex          0.000000
dzgroup      0.000000
dzclass      0.000000
num.co       0.000000
edu         17.888529
income      32.631822
scoma        0.000000
charges      1.919519
totcst       9.840426
totmcst     38.124422
avtisst      0.901943
race         0.439408
surv2m       0.000000
surv6m       0.000000
hday         0.000000
diabetes     0.000000
dementia     0.000000
ca           0.000000
prg2m       17.992599
prg6m       17.819149
dnr          0.312211
dnrday       0.312211
meanbp       0.011563
wblc         2.301110
hrt          0.011563
resp         0.011563
temp         0.011563
pafi        25.404718
alb         37.072155
bili        28.561517
crea         0.751619
sod          0.011563
ph          24.965310
glucose     49.398705
bun         47.756707
urine       53.283996
adlp        61.968085
adls        31.278908
adlsc        0.000000
death        0.000000
dtype: flo

3. Rimuovere le feature che presentano elevata correlazione con la variabile target e successivamente analizzare le possibili feature multicollineari rimanenti.

In [62]:
df_train = X_train.copy()
df_train['aps'] = y_train

corr_target = df_train.corr()['aps'].drop('aps').abs().sort_values(ascending=False)  #o corrwith() per calcolare direttamente la correlazione tra X e y
print(f"Correlazione assoluta con il target 'aps':\n{corr_target.head(50)}")

low_target_threshold = 0.05
low_corr_features = corr_target[corr_target < low_target_threshold].index.tolist()
print(f"\nFeature da rimuovere per bassa correlazione col target:\n{low_corr_features}")
X_train = X_train.drop(columns=low_corr_features)
X_test = X_test.drop(columns=low_corr_features)


corr_matrix = df_train.corr().abs()

#maschera per il triangolo superiore della matrice per evitare doppioni (A-B, B-A) e la diagonale (A-A)
upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

rel_threshold = 0.7
corr_pairs = upper_tri.unstack().dropna().sort_values(ascending=False)  #trasformiamo in una serie e ordiniamo i valori
rel_features = corr_pairs[corr_pairs > rel_threshold]
print(f"Coppie di feature multicollineari (R > {rel_threshold}):\n{rel_features}")

var_to_remove = ['surv6m', 'prg6m', 'adls', 'totcst']
X_train = X_train.drop(columns=var_to_remove, errors='ignore')
X_test = X_test.drop(columns=var_to_remove, errors='ignore')

print(f"\nNumero di feature rimanenti per l'addestramento: {X_train.shape[1]}")

Correlazione assoluta con il target 'aps':
surv2m      0.650936
avtisst     0.598127
surv6m      0.551590
dzclass     0.383056
prg2m       0.366753
charges     0.305728
dnr         0.296898
ca          0.295913
totcst      0.283629
scoma       0.280444
prg6m       0.264227
hday        0.256980
crea        0.241285
totmcst     0.234493
bili        0.232981
meanbp      0.165179
dzgroup     0.161140
death       0.156414
hrt         0.154015
alb         0.150338
dnrday      0.128008
pafi        0.124875
resp        0.118025
adlp        0.114926
adlsc       0.102529
wblc        0.093854
diabetes    0.085949
ph          0.085785
temp        0.077735
bun         0.070202
adls        0.060368
race        0.046748
sod         0.045391
age         0.037139
income      0.029468
glucose     0.028408
num.co      0.017447
urine       0.017256
sex         0.013453
dementia    0.012424
edu         0.004054
Name: aps, dtype: float64

Feature da rimuovere per bassa correlazione col target:
['race', 'sod

4. Utilizzare RandomForestClassifier per la classificazione e RandomForestRegressor per la regressione con i seguenti iperparametri(*)
    a. Classificatore
        i. criterion: “gini”, “log_loss”
        ii. min_samples_split: 2, 5, 10
        iii. max_features: “sqrt”, 5
    b. Regressore
        i. criterion: “squared_error”, “absolute_error”
        ii. min_samples_split: 2, 5, 10
        iii. max_features: “sqrt”, 5

In [67]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV

iperparam_grid = {
    'criterion': ["squared_error", "absolute_error"],
    'min_samples_split': [2, 5, 10],
    'max_features': ["sqrt", 5]
}

clf = RandomForestRegressor(random_state=42)
grid_search_clf = GridSearchCV(
    estimator=clf,
    param_grid=iperparam_grid,
    cv=5,  #cross-validation a 5 fold
    scoring='r2',  #per la regressione non si usa l'accuracy, ma il coefficiente di determinazione r^2 come metrica di valutazione
    n_jobs=-1
)

grid_search_clf.fit(X_train, y_train)
print(f"Search Grid - Migliori iperparametri: {grid_search_clf.best_params_}, Migliore punteggio R2: {grid_search_clf.best_score_:.4f}")

Search Grid - Migliori iperparametri: {'criterion': 'squared_error', 'max_features': 'sqrt', 'min_samples_split': 2}, Migliore punteggio R2: 0.6956


5. Valutare il regressore sul test set con la metrica R2, il classificatore binario con la curva ROC e la relativa AUC ed il classificatore multiclasse con le curve ROC e le AUC di ogni classe, ciascuna valutata in modalità one-vs-rest.

In [69]:
from sklearn.metrics import r2_score

best_regressor = grid_search_clf.best_estimator_  #miglior estimatore

#chiediamo al modello di fare le predizioni sui dati di test (che non ha mai visto)
y_pred_reg = best_regressor.predict(X_test)

#confrontiamo le predizioni (y_pred_reg) con la realtà (y_test) calcolando l'R2
r2 = r2_score(y_test, y_pred_reg)

print(f"Punteggio R2 sul Test Set: {r2:.4f}")

Punteggio R2 sul Test Set: 0.6618
